# CS661 Assignment 2 — Interactive Volume Visualization

This Jupyter Notebook provides an interactive interface for exploring a 3D scalar volume dataset from a turbulence/mixture simulation. The visualization uses:
- **PyVista**: To read the VTK ImageData (`.vti`) dataset efficiently and safely.
- **Plotly**: To render a 3D Isosurface plot alongside a 2D Histogram of the scalar values.
- **ipywidgets**: To provide a slider control for the isosurface value (`isoval`) and a reset button.

## Prerequisites

create a virtual environment
```bash
python -m venv .venv
```

### 2. Create and activate virtual environment
  ```bash
  python -m venv .venv
  .venv\Scripts\activate          # Windows(my system)

  # source .venv/bin/activate     # Linux/macOS
  ```

To run this dashboard locally, ensure you have Python installed along with the required libraries. You can install the dependencies using `pip`:

```bash
pip install numpy pyvista plotly ipywidgets vtk jupyter anywidget
```
it may take few minutes...

TO RUN IN VScode

select .venv just made as kernel and run cells

To run on browser using jupyter
```bash
jupyter notebook
```

In [1]:
import pyvista as pv
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
print("done installing")

done installing


In [2]:
# Data Loading & Exploration
# Read the mixture.vti file using PyVista.
# mixture.vti is a VTK ImageData structure.
grid = pv.read("mixture.vti")

# Extract the 3D scalar array "ImageFile" as a 1D numpy array.
# Grid dimensions: 75 x 75 x 75 = 421,875 points.
values = np.array(grid.point_data["ImageFile"])

# Extract coordinates: grid.points is an N x 3 array. 
# Points and values are in the same flattened order, so x[i], y[i], z[i] is the position of values[i].
points = grid.points
x, y, z = points[:, 0], points[:, 1], points[:, 2]

# Compute global min/max scalar values. We store these to configure the slider and colorscale ranges.
global_min = float(values.min())
global_max = float(values.max())

print("=== Dataset Information ===")
print(f"Scalar Array Name: 'ImageFile'")
print(f"Scalar range:      [{global_min:.6f}, {global_max:.6f}]")
print(f"Total Grid Points: {len(values)} (Dimensions: 75x75x75)")
print(f"Grid Bounds:")
print(f"  X: [{x.min():.2f}, {x.max():.2f}]")
print(f"  Y: [{y.min():.2f}, {y.max():.2f}]")
print(f"  Z: [{z.min():.2f}, {z.max():.2f}]")

=== Dataset Information ===
Scalar Array Name: 'ImageFile'
Scalar range:      [-0.993554, 0.432802]
Total Grid Points: 421875 (Dimensions: 75x75x75)
Grid Bounds:
  X: [0.00, 149.00]
  Y: [0.00, 149.00]
  Z: [0.00, 149.00]


In [3]:
# Cell 3: FigureWidget Initialization
# We combine the two Plotly plots (3D Isosurface and 2D Histogram) side-by-side using make_subplots.
# A go.FigureWidget is used instead of a standard go.Figure so we can update traces in place without page flicker.

fig = go.FigureWidget(make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "xy"}]],
    subplot_titles=("Interactive 3D Isosurface", "Scalar Values Frequency Histogram"),
    column_widths=[0.6, 0.4] # 60% for 3D plot, 40% for histogram for better visualization :)
))

# 1. Adding Isosurface trace to Row 1, Column 1:
# - isomin & isomax are both set to 0.0 initially to display a single isosurface shell at that exact value.
# - surface_count is set to 1.
# - caps are disabled for a cleaner visual layout.
# - colorscale is set to 'Plasma'. As we studied in class its good to use
# - cmin & cmax are fixed globally to [global_min, global_max] so the surface color shifts visually as isoval changes.
fig.add_trace(
    go.Isosurface(
        x=x, y=y, z=z,
        value=values,
        isomin=0.0,
        isomax=0.0,
        surface_count=1,
        colorscale="Plasma",
        cmin=global_min,
        cmax=global_max,
        caps=dict(x_show=False, y_show=False, z_show=False),
        colorbar=dict(title="Scalar", len=0.8, x=-0.08, y=0.5)
    ),
    row=1, col=1
)

# 2. Add Histogram trace to Row 1, Column 2:
# - Initially displays the full dataset.
# - Uses 50 bins for detailed representation.
fig.add_trace(
    go.Histogram(
        x=values,
        nbinsx=50,
        marker=dict(color="royalblue", line=dict(color="white", width=0.5))
    ),
    row=1, col=2
)

# Configure Layout Styling:
# Note: Column 1 is a 3D scene (has scene layout). Column 2 is the ONLY 2D xy plot,
# so layout references it using 'xaxis' and 'yaxis' (not 'xaxis2' and 'yaxis2').
fig.update_layout(
    title=dict(text="CS661 Assignment 2: Volume Visualization Dashboard", x=0.5),
    width=1000,
    height=550,
    showlegend=False,
    scene=dict(
        xaxis=dict(title="X", backgroundcolor="rgb(240, 240, 240)", gridcolor="white", showbackground=True),
        yaxis=dict(title="Y", backgroundcolor="rgb(240, 240, 240)", gridcolor="white", showbackground=True),
        zaxis=dict(title="Z", backgroundcolor="rgb(240, 240, 240)", gridcolor="white", showbackground=True)
    ),
    xaxis=dict(title="Vortex scalar values"),
    yaxis=dict(title="Frequency")
)

FigureWidget({
    'data': [{'caps': {'x': {'show': False}, 'y': {'show': False}, 'z': {'show': False}},
              'cmax': 0.43280163407325745,
              'cmin': -0.9935540556907654,
              'colorbar': {'len': 0.8, 'title': {'text': 'Scalar'}, 'x': -0.08, 'y': 0.5},
              'colorscale': [[0.0, '#0d0887'], [0.1111111111111111, '#46039f'],
                             [0.2222222222222222, '#7201a8'], [0.3333333333333333,
                             '#9c179e'], [0.4444444444444444, '#bd3786'],
                             [0.5555555555555556, '#d8576b'], [0.6666666666666666,
                             '#ed7953'], [0.7777777777777778, '#fb9f3a'],
                             [0.8888888888888888, '#fdca26'], [1.0, '#f0f921']],
              'isomax': 0.0,
              'isomin': 0.0,
              'scene': 'scene',
              'surface': {'count': 1},
              'type': 'isosurface',
              'uid': '88a9940e-0d60-4c85-86ab-bff38f2b957b',
              'va

In [4]:
# Cell 4: Widget Controls & Interactive Callbacks

# Function to update both plots based on the selected isovalue
def update_plots(isoval):
    # Update Isosurface: sets both bounds equal to the target isovalue to isolate the shell.
    # Modifying in place prevents figure recreation and eliminates flickering.
    fig.data[0].isomin = isoval
    fig.data[0].isomax = isoval
    
    # Update Histogram: filter values to keep only those within [isoval - 0.25, isoval + 0.25]
    mask = (values >= isoval - 0.25) & (values <= isoval + 0.25)
    filtered_values = values[mask]
    
    # Update the histogram trace with filtered values
    fig.data[1].x = filtered_values
    
    # Rescale the histogram's X-axis range to match this narrower window [isoval - 0.25, isoval + 0.25]
    fig.layout.xaxis.range = [isoval - 0.25, isoval + 0.25]

# Callback function triggered when the slider value changes
def on_slider_change(change):
    update_plots(change.new)

# Callback function triggered when the Reset button is clicked
def on_reset_click(button):
    # Reset slider back to 0.0 (triggers on_slider_change if value was different)
    slider.value = 0.0
    
    # Explicitly ensure elements revert to their initial default states
    # (This covers both the case where slider was already 0.0 and cases where updates must be overwritten)
    fig.data[0].isomin = 0.0
    fig.data[0].isomax = 0.0
    fig.data[1].x = values
    
    # Revert the histogram's X-axis range to show the full dataset's range
    fig.layout.xaxis.range = None
    fig.layout.xaxis.autorange = True

# Create ipywidgets UI elements
# Slider: FloatSlider spanning [global_min, global_max], formatted to 2 decimal places.
slider = widgets.FloatSlider(
    value=0.0,
    min=global_min,
    max=global_max,
    step=0.01,
    description='Isoval:',
    continuous_update=False, # Triggers update on release/pause to keep visualization smooth and avoid lags
    readout=True,
    readout_format='.2f',
    layout=widgets.Layout(width='60%')
)

# Reset Button: A button to restore the initial state
reset_btn = widgets.Button(
    description='Reset',
    button_style='info',
    icon='refresh',
    layout=widgets.Layout(width='10%')
)

# Wire the callbacks once as requested
slider.observe(on_slider_change, names='value')
reset_btn.on_click(on_reset_click)

# Layout controls side-by-side (HBox) and combine with the figure (VBox)
controls_layout = widgets.HBox([slider, reset_btn], layout=widgets.Layout(margin='10px 0px'))
app_layout = widgets.VBox([controls_layout, fig])

# Display the widget workspace
display(app_layout)